# AIOCR - Agentic Indic OCR: Example Inference Pipeline

This notebook demonstrates the full 7-agent pipeline for Indic document OCR.

## Pipeline:
1. Script Detection (CNN)
2. Image Restoration (ESRGAN/CLAHE)
3. Text Detection (YOLOv8/CRAFT)
4. Character Recognition (TrOCR)
5. LLM Correction (Claude)
6. Knowledge Retrieval (RAG + ChromaDB)
7. Output Formatting (JSON + Bounding Boxes)

In [ ]:
import sys
sys.path.insert(0, '..')

import asyncio
import base64
from PIL import Image
import io
import json

## 1. Via REST API (Recommended)

In [ ]:
import requests

# Start the server: uvicorn app.main:app --reload
BASE_URL = 'http://localhost:8000/api/v1'

# Example: Script Detection only
with open('../tests/sample_devanagari.png', 'rb') as f:
    response = requests.post(
        f'{BASE_URL}/detect-script/',
        files={'file': ('sample.png', f, 'image/png')}
    )

print(json.dumps(response.json(), indent=2, ensure_ascii=False))

In [ ]:
# Example: Full Pipeline
with open('../tests/sample_devanagari.png', 'rb') as f:
    response = requests.post(
        f'{BASE_URL}/full-pipeline/',
        files={'file': ('sample.png', f, 'image/png')},
        params={
            'enable_restoration': True,
            'enable_rag': True,
            'include_annotated_image': True,
        }
    )

result = response.json()
print('Script:', result['script'])
print('Confidence:', result['overall_confidence'])
print('\nRaw OCR:', result['raw_text'])
print('\nCorrected:', result['corrected_text'])
print('\nCorrections Made:')
for c in result['corrections_made']:
    print(' -', c)
print('\nReasoning:', result['reasoning'])

In [ ]:
# Display annotated image if included
if result.get('annotated_image_base64'):
    img_bytes = base64.b64decode(result['annotated_image_base64'])
    img = Image.open(io.BytesIO(img_bytes))
    display(img)

## 2. Direct Python Agent Pipeline

In [ ]:
import os
os.environ['MODEL_DIR'] = '../saved_models'
os.environ['ANTHROPIC_API_KEY'] = 'your-api-key-here'

from agents.orchestrator import AIRCOrchestrator

# Load test image
with open('../tests/sample_devanagari.png', 'rb') as f:
    image_bytes = f.read()

orchestrator = AIRCOrchestrator()

result = await orchestrator.run(
    image_bytes=image_bytes,
    request_id='notebook-test-001',
    enable_restoration=True,
    enable_rag=False,  # Disable RAG if ChromaDB not running
    include_annotated_image=True,
)

print(json.dumps({
    k: v for k, v in result.items()
    if k not in ('annotated_image_base64', 'restored_image_base64')
}, indent=2, ensure_ascii=False))

## 3. Script Detection Only (Fast Path)

In [ ]:
from models.cnn_classifier import ScriptClassifier
import numpy as np
from PIL import Image

classifier = ScriptClassifier.get_instance()
print('Available models:', classifier.available_models)

with open('../tests/sample_devanagari.png', 'rb') as f:
    image_bytes = f.read()

script, confidence, model_used = classifier.predict(image_bytes, model_name='ensemble')
print(f'Script: {script}')
print(f'Confidence: {confidence:.4f}')
print(f'Model: {model_used}')

## 4. Agent Status Timeline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Visualize agent processing times from result
agent_statuses = result.get('agent_statuses', [])

names = [s['agent_name'] for s in agent_statuses]
times = [s.get('processing_time_ms', 0) or 0 for s in agent_statuses]
statuses = [s['status'] for s in agent_statuses]

colors = {'completed': '#4CAF50', 'failed': '#f44336', 'skipped': '#9E9E9E', 'fallback': '#FF9800'}
bar_colors = [colors.get(s, '#2196F3') for s in statuses]

plt.figure(figsize=(12, 5))
bars = plt.barh(names, times, color=bar_colors)
plt.xlabel('Processing Time (ms)')
plt.title('AIOCR Agent Pipeline - Processing Times')
plt.tight_layout()
plt.show()